In [ ]:
import sys
import os
import urllib.request
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.linalg import eig
from scipy.stats import spearmanr
from kneed import KneeLocator
from scipy.spatial.distance import pdist, squareform

# Reproducible project paths and output locations

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Run this notebook from inside the DELVE repository.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))



































from project_utils import DATA_DIR, TABLES_DIR, configure_plots, save_figure, ensure_output_dirs
from functions import Kernel_matrix, LG_sym, calc_differential_vec, diffusion_map

configure_plots()
ensure_output_dirs()


# Ensure imports work whether Jupyter runs from project root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:bold'

In [ ]:
def Kernel_matrix(df, epsilon, coifman_lafon = False, alpha = 1):
    n_cells = df.shape[0]
    #euclidian distance matrix:
    D = squareform(pdist(df,'euclidean'))
    dist_sort = np.sort(D,axis = 1)
    sigmas = dist_sort[:, epsilon]#np.mean(dist_sort[:, 1:epsilon+1], axis=1)
    Sig = np.outer(sigmas,sigmas)#sigmas*sigmas
    K = np.exp(-(D**2)/Sig)

    if coifman_lafon:
        q = K.sum(axis=1)
    
        # Density correction
        q_alpha = q ** (-alpha)
        K_tilde = q_alpha[:, None] * K * q_alpha[None, :]

        return K_tilde
    else:
        return K
    

In [ ]:
# Settings
n_samples = 2000
random_state = 42
shape_value = 1          # 0=square, 1=ellipse, 2=heart
A_scale_class = 4
B_orientation_class = 0

data_file = DATA_DIR / "dsprites_ndarray_co1sh3sc6or40x32y32_64x64.hdf5"
download_url = (
    "https://github.com/google-deepmind/dsprites-dataset/raw/master/"
    "dsprites_ndarray_co1sh3sc6or40x32y32_64x64.hdf5"
)

if not os.path.exists(data_file):
    urllib.request.urlretrieve(download_url, data_file)


# Latent variables
rng = np.random.default_rng(random_state)

shape_class = np.full(n_samples, shape_value)
shared_y_class = rng.integers(0, 32, n_samples)

A_x_class = 12 #rng.integers(0, 32, n_samples)
A_orientation_class = rng.integers(0, 40, n_samples)
A_scale_class = rng.integers(1, 6, n_samples)

B_x_class = rng.integers(0, 32, n_samples)
B_scale_class = rng.integers(1, 6, n_samples)


# dSprites indices:
# shape, scale, orientation, x-position, y-position
A_indices = (
    (((shape_class * 6 + A_scale_class) * 40 + A_orientation_class) * 32
     + A_x_class) * 32
    + shared_y_class
)

B_indices = (
    (((shape_class * 6 + B_scale_class) * 40 + B_orientation_class) * 32
     + B_x_class) * 32
    + shared_y_class
)


# Load selected images
with h5py.File(data_file, "r") as data:
    images = data["imgs"]

    A_unique, A_inverse = np.unique(A_indices, return_inverse=True)
    B_unique, B_inverse = np.unique(B_indices, return_inverse=True)

    XA_images = images[A_unique][A_inverse]
    XB_images = images[B_unique][B_inverse]


# Flatten for DELVE
XA = XA_images.reshape(n_samples, -1).astype(np.float32)
XB = XB_images.reshape(n_samples, -1).astype(np.float32)


# Numerical latent values
shared_y = shared_y_class / 31
A_x = A_x_class / 31
B_x = B_x_class / 31

A_orientation = 2 * np.pi * A_orientation_class / 40
B_scale = 0.5 + 0.1 * B_scale_class

In [ ]:
orientation_gt = [np.cos(2 * A_orientation),
np.sin(2 * A_orientation)]

In [ ]:
# Core operators/eigenvectors for all methods on the representative sample
P1, Q1, K1 = diffusion_map(XA, adaptive=900)
P2, Q2, K2 = diffusion_map(XB, adaptive=900)

L1, d1, v1 = LG_sym(K1)
L2, d2, v2 = LG_sym(K2)

In [ ]:
kl = KneeLocator(np.arange(len(d1)), d1, curve="convex", direction="decreasing",S=0.5)
tau1 = kl.knee
kl = KneeLocator(np.arange(len(d2)), d2, curve="convex", direction="decreasing",S=0.5)
tau2 = kl.knee
print(tau1)
print(tau2)

In [ ]:
Q_21, _, u1 = calc_differential_vec(L2, v1, tau1, Q=True)
Q_12, _, u2 = calc_differential_vec(L1, v2, tau2, Q=True)

S = P2 @ Q1 + P1 @ Q2
D = P2 @ Q1 - P1 @ Q2

_, _ = eig(S)  # computed in original workflow; not used directly in final outputs
ea_vals, va = eig(D)
VA_imag = np.imag(va[:, np.argsort(np.imag(ea_vals))[::-1]])
VA_real = np.real(va[:, np.argsort(np.real(ea_vals))[::-1]])

g1 = np.diag(np.sum(K1, axis=1)) - K1
g2 = np.diag(np.sum(K2, axis=1)) - K2

m1 = g1 + 1e-6 * np.eye(g1.shape[0])
m2 = g2 + 1e-6 * np.eye(g2.shape[0])

fk1 = np.linalg.inv(m1 + m2) @ m1
fk2 = np.linalg.inv(m1 + m2) @ m2

fk_vals_1, eig_vec_fk_1 = eig(fk1)
fk_vals_2, eig_vec_fk_2 = eig(fk2)

eig_vec_fk_1 = eig_vec_fk_1[:, np.argsort(fk_vals_1)[::-1]]
eig_vec_fk_2 = eig_vec_fk_2[:, np.argsort(fk_vals_2)[::-1]]

# Shnitzer et al.

In [ ]:
S = P2 @ Q1 + P1 @ Q2
D = P2 @ Q1 - P1 @ Q2
_, _ = eig(S)
ea_vals, va = eig(D)

va_imag = np.imag(va[:, np.argsort(np.imag(ea_vals))[::-1]])
va_real = np.real(va[:, np.argsort(np.real(ea_vals))[::-1]])

In [ ]:
#FKT
g1 = np.diag(np.sum(K1, axis=0)) - K1
g2 = np.diag(np.sum(K2, axis=0)) - K2

m1 = g1 + 1e-6 * np.eye(g1.shape[0])
m2 = g2 + 1e-6 * np.eye(g2.shape[0])

fk1 = np.linalg.inv(m1 + m2) @ m1
fk2 = np.linalg.inv(m1 + m2) @ m2

fk_values_1, eig_vec_fk_1 = eig(fk1)
fk_values_2, eig_vec_fk_2 = eig(fk2)

eig_vec_fk_1 = eig_vec_fk_1[:, np.argsort(fk_values_1)[::-1]]
eig_vec_fk_2 = eig_vec_fk_2[:, np.argsort(fk_values_2)[::-1]]

# DELVE

In [ ]:
np.corrcoef(orientation_gt[0],u2[:,0])
np.corrcoef(A_scale_class,u2[:,0])

In [ ]:
np.corrcoef(B_x,u1[:,0])

In [ ]:
# L_shared = Q_21@Q_12 + Q_12@Q_21
L_shared = L1@L2 + L2@L1

# L_shared = L1@v2@v2.T @ L2@v1@v1.T + L2@v1@v1.T @ L1@v2@v2.T 

ds, v_shared = np.linalg.eigh(L_shared)
idx_s = np.argsort(ds)[::-1]
v_shared = v_shared[:,idx_s]

In [ ]:
V_2 = np.concatenate((v_shared[:,:6], u2[:,:1]), axis = 1)
K_V2 = Kernel_matrix(V_2,150,True)

L_V2, d_VB, v_V2 = LG_sym(K_V2)

# tau_V2, u_V2, scores_V2 = choose_tau(L_own=Q_12,L_other=L_V2,v_other=v_V2)

kl = KneeLocator(np.arange(len(d_VB)), d_VB, curve="convex", direction="decreasing",S=0.25)
tauV2 = kl.knee
print(tauV2)

L1_nr, s_nr2, u2_nr = calc_differential_vec(L1, v_V2, tauV2,"yes")

In [ ]:
np.corrcoef(orientation_gt[0], u2_nr[:,0])

In [ ]:
V_1 = np.concatenate((v_shared[:,:6], u1[:,0:1]), axis = 1)
K_V1 = Kernel_matrix(V_1,150, False)

L_V1, d_VA, v_V1 = LG_sym(K_V1)

# tau_V1, u_V1, scores_V1 = choose_tau(L_own=L2,L_other=L_V1,v_other=v_V1)

kl = KneeLocator(np.arange(len(d_VA)), d_VA, curve="convex", direction="decreasing",S=0.25)
tauV1 = kl.knee
print(tauV1)

L2_nr, s_nr, u1_nr = calc_differential_vec(L2,v_V1,tauV1,"yes")

In [ ]:
np.corrcoef(B_scale,u1_nr[:,0])

## Repeated experiment (100 runs)

The table reports mean (SD) absolute correlations for the leading differential vectors.

In [ ]:
def run_once(seed, images):
    rng = np.random.default_rng(seed)

    shape = np.full(n_samples, shape_value)
    y = rng.integers(0, 32, n_samples)
    A_rot = rng.integers(0, 40, n_samples)
    A_scale = rng.integers(1, 6, n_samples)
    B_x = rng.integers(0, 32, n_samples)
    B_scale = rng.integers(1, 6, n_samples)

    A_idx = ((((shape*6 + A_scale)*40 + A_rot)*32 + A_x_class)*32 + y)
    B_idx = ((((shape*6 + B_scale)*40 + B_orientation_class)*32 + B_x)*32 + y)

    A_u, A_inv = np.unique(A_idx, return_inverse=True)
    B_u, B_inv = np.unique(B_idx, return_inverse=True)

    XA = images[A_u][A_inv].reshape(n_samples, -1).astype(np.float32)
    XB = images[B_u][B_inv].reshape(n_samples, -1).astype(np.float32)

    P1, Q1, K1 = diffusion_map(XA, adaptive=900)
    P2, Q2, K2 = diffusion_map(XB, adaptive=900)
    L1, d1, v1 = LG_sym(K1)
    L2, d2, v2 = LG_sym(K2)

    kl = KneeLocator(np.arange(len(d1)), d1, curve="convex", direction="decreasing",S=0.5)
    tau1 = kl.knee
    kl = KneeLocator(np.arange(len(d2)), d2, curve="convex", direction="decreasing",S=0.5)
    tau2 = kl.knee

    Q_21, _, u1 = calc_differential_vec(L2, v1, tau1, Q=True)
    Q_12, _, u2 = calc_differential_vec(L1, v2, tau2, Q=True)

    _, v_shared = np.linalg.eigh(L1@L2 + L2@L1)
    v_shared = v_shared[:, ::-1]

    V2 = np.c_[v_shared[:, :3], u2[:, 0]]
    L_V2, d_V2, v_V2 = LG_sym(Kernel_matrix(V2, 150, True))
    kl = KneeLocator(np.arange(len(d_V2)), d_V2, curve="convex", direction="decreasing",S=0.25)
    tauV2 = kl.knee
    _, _, u2_nr = calc_differential_vec(L1, v_V2, tauV2, "yes")

    V1 = np.c_[v_shared[:, :6], u1[:, 0]]
    L_V1, d_V1, v_V1 = LG_sym(Kernel_matrix(V1, 150, False))
    kl = KneeLocator(np.arange(len(d_V1)), d_V1, curve="convex", direction="decreasing",S=0.25)
    tauV1 = kl.knee
    _, _, u1_nr = calc_differential_vec(L2, v_V1, tauV1, "yes")

    phi = 2*np.pi*A_rot/40
    orientation_score = max(
        abs(spearmanr(u2_nr[:, 0], np.cos(2*phi)).statistic),
        abs(spearmanr(u2_nr[:, 0], np.sin(2*phi)).statistic),
    )

    # Shnitzer et al.
    S = P2 @ Q1 + P1 @ Q2
    D = P2 @ Q1 - P1 @ Q2
    _, _ = eig(S)
    ea_vals, va = eig(D)

    va_imag = np.imag(va[:, np.argsort(np.imag(ea_vals))[::-1]])
    va_real = np.real(va[:, np.argsort(np.real(ea_vals))[::-1]])

    #FKT
    g1 = np.diag(np.sum(K1, axis=0)) - K1
    g2 = np.diag(np.sum(K2, axis=0)) - K2

    m1 = g1 + 1e-6 * np.eye(g1.shape[0])
    m2 = g2 + 1e-6 * np.eye(g2.shape[0])

    fk1 = np.linalg.inv(m1 + m2) @ m1
    fk2 = np.linalg.inv(m1 + m2) @ m2

    fk_values_1, eig_vec_fk_1 = eig(fk1)
    fk_values_2, eig_vec_fk_2 = eig(fk2)

    eig_vec_fk_1 = eig_vec_fk_1[:, np.argsort(fk_values_1)[::-1]]
    eig_vec_fk_2 = eig_vec_fk_2[:, np.argsort(fk_values_2)[::-1]]

    # Targets
    A_orientation_targets = [np.cos(2 * phi), np.sin(2 * phi)]
    
    # FKT: modality A
    fkt_A = np.real(eig_vec_fk_2[:, 0])
    corr_fkt_A_scale = abs(spearmanr(fkt_A, A_scale).statistic)
    corr_fkt_A_orientation = max(
        abs(spearmanr(fkt_A, t).statistic)
        for t in A_orientation_targets
    )
    
    # FKT: modality B
    fkt_B = np.real(eig_vec_fk_1[:, 0])
    corr_fkt_B_x = abs(spearmanr(fkt_B, B_x).statistic)
    corr_fkt_B_scale = abs(spearmanr(fkt_B, B_scale).statistic)
    
    # Shnitzer: leading real vector
    sh_real = va_real[:, 0]
    corr_sh_real_A_scale = abs(spearmanr(sh_real, A_scale).statistic)
    corr_sh_real_A_orientation = max(
        abs(spearmanr(sh_real, t).statistic)
        for t in A_orientation_targets
    )
    corr_sh_real_B_x = abs(spearmanr(sh_real, B_x).statistic)
    corr_sh_real_B_scale = abs(spearmanr(sh_real, B_scale).statistic)
    
    # Shnitzer: leading imaginary vector
    sh_imag = va_imag[:, 0]
    corr_sh_imag_A_scale = abs(spearmanr(sh_imag, A_scale).statistic)
    corr_sh_imag_A_orientation = max(abs(spearmanr(sh_imag, t).statistic)
        for t in A_orientation_targets
    )
    corr_sh_imag_B_x = abs(spearmanr(sh_imag, B_x).statistic)
    corr_sh_imag_B_scale = abs(spearmanr(sh_imag, B_scale).statistic)
    
    return [
    # DELVE
    abs(spearmanr(u2[:, 0], A_scale).statistic),
    orientation_score,
    abs(spearmanr(u1[:, 0], B_x).statistic),
    abs(spearmanr(u1_nr[:, 0], B_scale).statistic),

    # FKT leading vectors
    corr_fkt_A_scale,
    corr_fkt_A_orientation,
    corr_fkt_B_x,
    corr_fkt_B_scale,

    # Shnitzer leading real vector
    corr_sh_real_A_scale,
    corr_sh_real_A_orientation,
    corr_sh_real_B_x,
    corr_sh_real_B_scale,

    # Shnitzer leading imaginary vector
    corr_sh_imag_A_scale,
    corr_sh_imag_A_orientation,
    corr_sh_imag_B_x,
    corr_sh_imag_B_scale,
]


B = 500

with h5py.File(data_file, "r") as data:
    results = np.array([run_once(seed, data["imgs"]) for seed in range(B)])

In [ ]:
def run_once_b_scale(seed, images):
    rng = np.random.default_rng(seed)

    shape = np.full(n_samples, shape_value)
    y = rng.integers(0, 32, n_samples)
    A_rot = rng.integers(0, 40, n_samples)
    A_scale = rng.integers(1, 6, n_samples)
    B_x = rng.integers(0, 32, n_samples)
    B_scale = rng.integers(1, 6, n_samples)

    A_idx = ((((shape*6 + A_scale)*40 + A_rot)*32 + A_x_class)*32 + y)
    B_idx = ((((shape*6 + B_scale)*40 + B_orientation_class)*32 + B_x)*32 + y)

    A_u, A_inv = np.unique(A_idx, return_inverse=True)
    B_u, B_inv = np.unique(B_idx, return_inverse=True)

    XA = images[A_u][A_inv].reshape(n_samples, -1).astype(np.float32)
    XB = images[B_u][B_inv].reshape(n_samples, -1).astype(np.float32)

    P1, Q1, K1 = diffusion_map(XA, adaptive=900)
    P2, Q2, K2 = diffusion_map(XB, adaptive=900)
    L1, d1, v1 = LG_sym(K1)
    L2, d2, v2 = LG_sym(K2)

    kl = KneeLocator(np.arange(len(d1)), d1, curve="convex", direction="decreasing",S=0.5)
    tau1 = kl.knee
    kl = KneeLocator(np.arange(len(d2)), d2, curve="convex", direction="decreasing",S=0.5)
    tau2 = kl.knee

    Q_21, _, u1 = calc_differential_vec(L2, v1, tau1, Q=True)
    Q_12, _, u2 = calc_differential_vec(L1, v2, tau2, Q=True)

    _, v_shared = np.linalg.eigh(L1@L2 + L2@L1)
    v_shared = v_shared[:, ::-1]

    V1 = np.c_[v_shared[:, :6], u1[:, 0]]
    L_V1, d_V1, v_V1 = LG_sym(Kernel_matrix(V1, 150, False))
    kl = KneeLocator(np.arange(len(d_V1)), d_V1, curve="convex", direction="decreasing",S=0.25)
    tauV1 = kl.knee
    _, _, u1_nr = calc_differential_vec(L2, v_V1, tauV1, "yes")

    phi = 2*np.pi*A_rot/40
    orientation_score = max(
        abs(spearmanr(u2_nr[:, 0], np.cos(2*phi)).statistic),
        abs(spearmanr(u2_nr[:, 0], np.sin(2*phi)).statistic),
    )

  
        
    return [
    # DELVE
    abs(spearmanr(u1_nr[:, 0], B_scale).statistic),
        abs(np.corrcoef(u1_nr[:, 0], B_scale)[0,1]),

    
]


B_b = 10

with h5py.File(data_file, "r") as data:
    results_b_scale = np.array([run_once_b_scale(seed, data["imgs"]) for seed in range(B_b)])

In [ ]:
np.mean(results_b_scale[:,0])

In [ ]:
summary = pd.DataFrame(index=[
    r"$A$: scale",
    r"$A$: orientation",
    r"$B$: x-position",
    r"$B$: scale",
])

summary["DELVE"] = [
    f"{results[:,0].mean():.3f} ({results[:,0].std():.3f})",
    f"{results[:,1].mean():.3f} ({results[:,1].std():.3f})",
    f"{results[:,2].mean():.3f} ({results[:,2].std():.3f})",
    f"{results[:,3].mean():.3f} ({results[:,3].std():.3f})",
]

summary["FKT"] = [
    f"{results[:,4].mean():.3f} ({results[:,4].std():.3f})",
    f"{results[:,5].mean():.3f} ({results[:,5].std():.3f})",
    f"{results[:,6].mean():.3f} ({results[:,6].std():.3f})",
    f"{results[:,7].mean():.3f} ({results[:,7].std():.3f})",
]

summary["Shnitzer (real)"] = [
    f"{results[:,8].mean():.3f} ({results[:,8].std():.3f})",
    f"{results[:,9].mean():.3f} ({results[:,9].std():.3f})",
    f"{results[:,10].mean():.3f} ({results[:,10].std():.3f})",
    f"{results[:,11].mean():.3f} ({results[:,11].std():.3f})",
]

summary["Shnitzer (imag)"] = [
    f"{results[:,12].mean():.3f} ({results[:,12].std():.3f})",
    f"{results[:,13].mean():.3f} ({results[:,13].std():.3f})",
    f"{results[:,14].mean():.3f} ({results[:,14].std():.3f})",
    f"{results[:,15].mean():.3f} ({results[:,15].std():.3f})",
]

display(summary)

summary.to_latex(
    TABLES_DIR / "dsprites_alg2_summary.tex",
    escape=False,
)

In [ ]:
# Six representative paired observations
idx = rng.choice(n_samples, 4, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(10, 5))

for j, i in enumerate(idx):
    axes[0, j].imshow(XA_images[i], cmap="gray")
    axes[0, j].set_title(
        f"y={shared_y_class[i]}, scale={A_scale_class[i]}, rot={A_orientation_class[i]}",
        fontsize=17,
    )

    axes[1, j].imshow(XB_images[i], cmap="gray")
    axes[1, j].set_title(
        f"y={shared_y_class[i]}, x={B_x_class[i]}, scale={B_scale_class[i]}",
        fontsize=17,
    )

    axes[0, j].axis("off")
    axes[1, j].axis("off")

fig.text(
    0.01, 0.73, r"$X^A$",
    fontsize=30,
    va="center",
    ha="center"
)

fig.text(
    0.01, 0.27, r"$X^B$",
    fontsize=30,
    va="center",
    ha="center"
)

plt.tight_layout(rect=[0.04, 0, 1, 1])
save_figure(
    "dsprites_examples.pdf",
    bbox_inches="tight",
    pad_inches=0.05,
)
plt.show()